# Policy / Transform 평가 가능성

## 분석 목적
Policy 강화의 Privacy–Utility–Runtime tradeoff를 현재 Bundle로 식별 가능한지 확인한다.

## 가설
H0: 같은 Case·Model·sampling 조건에서 Policy/Transform 개입의 결과 차이가 없다.

## 사용할 변수
policy_snapshot_digest, model_profile_id, final_action 및 현재 존재하는 runtime 변수

## 통계기법 선택 이유
먼저 처치 조건과 대조군, privacy/utility outcome, pairing 설계를 확인한다. 모델 profile은 policy treatment가 아니다. v1에는 한 policy snapshot만 있어 인과 검정하지 않는다.

## 해석 기준
NOT EVALUABLE WITH CURRENT BUNDLE. Final action은 실행 결과이지 랜덤 처치가 아니다. 없는 Utility, privacy 점수나 transform 강도를 생성하지 않는다. 추가 실험을 명시한다.

기본 입력은 **합성 Consumer 시험 fixture**이다. 실제 Bundle은 `ADP_AI_BUNDLE_SOURCE`로 지정한다. Case 독립성/대칭성은 자동 추정하지 않는다.


In [ ]:
import os
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "02_ai/src").is_dir())
sys.path.insert(0, str(ROOT / "02_ai/src"))
from adp_da.bundle_analysis import analyze_bundle  # noqa: E402
from adp_da.bundle_dataset import execution_dataframe  # noqa: E402
from adp_da.bundle_loader import load_bundle  # noqa: E402

source = os.environ.get("ADP_AI_BUNDLE_SOURCE")
synthetic = source is None or os.environ.get("ADP_AI_SYNTHETIC") == "1"
if source is None:
    source = str(ROOT / "02_ai/tests/fixtures/evaluation_bundle.synthetic.json")
bundle, metadata = load_bundle(
    source, ROOT / "02_ai/data/interim/ai_evaluation/raw",
    evaluation_run_id=os.environ.get("ADP_AI_EVALUATION_RUN_ID"),
    token=os.environ.get("ADP_BE_TOKEN"),
    local_admin_user_id=os.environ.get("ADP_BE_LOCAL_ADMIN_USER_ID"),
    local_admin_roles=os.environ.get("ADP_BE_LOCAL_ADMIN_ROLES"),
)
frame = execution_dataframe(bundle)
artifacts = analyze_bundle(
    frame, synthetic=synthetic,
    independent_cases=os.environ.get("ADP_AI_INDEPENDENT_CASES") == "1",
    symmetric_differences=os.environ.get("ADP_AI_SYMMETRIC_DIFFERENCES") == "1",
)
print("SYNTHETIC FIXTURE — SOFTWARE VALIDATION ONLY" if synthetic else "USER-SUPPLIED BUNDLE")
display({k: bundle["manifest"][k] for k in ("bundle_id", "content_digest", "execution_count")})


In [ ]:
display(artifacts["policy_effect_analysis"])
display(artifacts["evaluation_summary"]["decision_candidates"])
display(artifacts["evaluation_summary"]["handoff_gaps"])
